# Multi-Regional AeroMAPS Tutorial — EU Domestic & International

This notebook is a minimal introduction to multi-regional modeling in AeroMAPS.
It runs two regions in parallel — **EU Domestic** (`EU_DOM`) and **EU International** (`EU_INT`) —
and shows how to create, run, and visualize a multi-regional process.

The same pattern scales to any number of regions (see `examples_multi_regional_simple.ipynb`).

## Imports

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt

from aeromaps import create_multi_regional_process
from aeromaps.utils.functions import create_partitioning

figures_dir = Path("figures")
figures_dir.mkdir(parents=True, exist_ok=True)

%matplotlib widget

## 1. Prepare Region Data

`create_partitioning` reads the AeroSCOPE CSV for each region and generates
the `partitioning_updated_inputs.json` file that initialises AeroMAPS traffic data.
Run this once (or whenever the CSV data changes).

In [ ]:
region_folders = ["region_eu_dom", "region_eu_int"]
data_path = Path("data")

for folder_name in region_folders:
    folder_path = data_path / folder_name
    csv_file = folder_path / "dataframe_aeromaps.csv"
    create_partitioning(file=str(csv_file), path=str(folder_path))
    print(f"✓ Prepared: {folder_name}")

## 2. Create and Run the Multi-Regional Process — both execution modes

Two near-identical configuration files describe the same two regions and differ **only** in
`execution_mode`:

- `regionalisation_europe_separate_processes.yaml` — each region is solved independently, then aggregated.
- `regionalisation_europe_unified_mda.yaml` — all regional disciplines are solved together in a single MDAChain.

We run **both** so the next section can check they produce the same results. The two modes should
always agree; a mismatch points to a bug in how one mode populates `process.data`.

In [ ]:
# Run the same scenario in both execution modes (the two configs differ only in
# `execution_mode`). Running both lets the next cell cross-check that they agree.
configs = {
    "separate_processes": "regionalisation_europe_separate_processes.yaml",
    "unified_mda": "regionalisation_europe_unified_mda.yaml",
}

processes = {}
for mode, config_file in configs.items():
    p = create_multi_regional_process(
        configuration_file=config_file,
        disable_execution_statistics=True,
    )
    p.compute(parallel=False)
    processes[mode] = p
    print(f"✓ {mode:<20s} | mode={p.execution_mode} | regions={p.list_regions()}")

# Use one of the two for the exploration / plots below
process = processes["unified_mda"]
print("\n✓ Computation complete for both modes")

## 3. Cross-Mode Consistency Check

`separate_processes` and `unified_mda` must produce the same outputs for the same scenario.
This cell compares every `vector_outputs` / `climate_outputs` column and every `float_outputs`
key between the two modes and **raises** if they diverge — a guardrail against regressions such
as regional columns silently missing from one mode.

In [ ]:
import numpy as np

sep = processes["separate_processes"]
uni = processes["unified_mda"]


def compare_frames(a, b, label, atol=1e-6, rtol=1e-6):
    """Return human-readable differences between two output DataFrames."""
    problems = []
    cols_a, cols_b = set(a.columns), set(b.columns)
    only_sep, only_uni = sorted(cols_a - cols_b), sorted(cols_b - cols_a)
    if only_sep:
        problems.append(
            f"{label}: {len(only_sep)} column(s) only in separate_processes, e.g. {only_sep[:5]}"
        )
    if only_uni:
        problems.append(
            f"{label}: {len(only_uni)} column(s) only in unified_mda, e.g. {only_uni[:5]}"
        )
    mismatched = []
    for col in sorted(cols_a & cols_b):
        va = a[col].to_numpy(dtype=float)
        vb = b[col].reindex(a.index).to_numpy(dtype=float)
        if not np.allclose(va, vb, atol=atol, rtol=rtol, equal_nan=True):
            mismatched.append(col)
    if mismatched:
        problems.append(
            f"{label}: {len(mismatched)} column(s) differ numerically, e.g. {mismatched[:5]}"
        )
    return problems


def compare_floats(a, b, label, atol=1e-6, rtol=1e-6):
    """Return human-readable differences between two float_outputs dicts."""
    problems = []
    keys_a, keys_b = set(a), set(b)
    only_sep, only_uni = sorted(keys_a - keys_b), sorted(keys_b - keys_a)
    if only_sep:
        problems.append(
            f"{label}: {len(only_sep)} key(s) only in separate_processes, e.g. {only_sep[:5]}"
        )
    if only_uni:
        problems.append(f"{label}: {len(only_uni)} key(s) only in unified_mda, e.g. {only_uni[:5]}")
    mismatched = []
    for k in sorted(keys_a & keys_b):
        va, vb = a[k], b[k]
        if isinstance(va, (int, float)) and isinstance(vb, (int, float)):
            ok = np.isclose(va, vb, atol=atol, rtol=rtol, equal_nan=True)
        else:
            ok = va == vb
        if not ok:
            mismatched.append(k)
    if mismatched:
        problems.append(
            f"{label}: {len(mismatched)} key(s) differ numerically, e.g. {mismatched[:5]}"
        )
    return problems


problems = []
problems += compare_frames(sep.data["vector_outputs"], uni.data["vector_outputs"], "vector_outputs")
problems += compare_frames(
    sep.data["climate_outputs"], uni.data["climate_outputs"], "climate_outputs"
)
problems += compare_floats(sep.data["float_outputs"], uni.data["float_outputs"], "float_outputs")

if problems:
    print("⚠️  The two execution modes DISAGREE:\n")
    for problem in problems:
        print("  -", problem)
    raise AssertionError(
        "separate_processes and unified_mda produced different results (see above)."
    )

print(
    "✓ separate_processes and unified_mda agree on all outputs:\n"
    f"    vector_outputs : {sep.data['vector_outputs'].shape[1]} columns\n"
    f"    climate_outputs: {sep.data['climate_outputs'].shape[1]} columns\n"
    f"    float_outputs  : {len(sep.data['float_outputs'])} keys"
)

## 4. Explore Outputs

All outputs are stored in `process.data["vector_outputs"]` with region-prefixed column names
(e.g. `EU_DOM:co2_emissions_including_energy`) plus aggregated `overall:` columns.

In [ ]:
vo = process.data["vector_outputs"]

print("Available regions:", process.list_regions())
print("\nCO₂ emissions incl. energy (Mt) at key years:")
cols = [
    "EU_DOM:co2_emissions_including_energy",
    "EU_INT:co2_emissions_including_energy",
    "overall:co2_emissions_including_energy",
]
print((vo[cols].loc[[2019, 2030, 2050]] / 1e9).round(1).to_string())

In [ ]:
process.get_regional_process("EU_DOM")

## 5. Visualize — CO₂ Emissions by Region

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))

series = [
    ("EU_DOM:co2_emissions_including_energy", "EU Domestic", "#4daf4a", "--"),
    ("EU_INT:co2_emissions_including_energy", "EU International", "#2171b5", ":"),
    ("overall:co2_emissions_including_energy", "EU DOM + INT combined", "black", "-"),
]

for col, label, color, ls in series:
    if col in vo.columns:
        (vo[col] / 1e9).plot(ax=ax, label=label, color=color, linestyle=ls, linewidth=1.8)

ax.set_xlabel("Year")
ax.set_ylabel("CO₂ Emissions (Mt)")
ax.set_xlim(2019, 2050)
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(figures_dir / "eu_co2_emissions.pdf", format="pdf", bbox_inches="tight")
plt.show()

## 6. Visualize — Emissions Decomposition (Waterfall)

Fill-between areas show the contribution of each mitigation lever
(fleet renewal, future aircraft, operations, low-carbon energy, CORSIA offsets).

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5), sharey=False)

fills = [
    (
        "co2_emissions_2019technology",
        "co2_emissions_including_aircraft_efficiency",
        "#FFE082",
        0.55,
        "Fleet Renewal",
    ),
    (
        "co2_emissions_including_aircraft_efficiency",
        "co2_emissions_including_load_factor",
        "#FFC107",
        0.55,
        "Operations",
    ),
    (
        "co2_emissions_including_load_factor",
        "co2_emissions_including_energy",
        "#66BB6A",
        0.50,
        "Low-carbon energy",
    ),
]

for ax, region_id in zip(axes, ["EU_DOM", "EU_INT"]):
    years = vo.index

    s_bau = vo.get(f"{region_id}:co2_emissions_2019technology")
    s_net_col = f"{region_id}:co2_emissions_including_energy"
    s_offset_col = f"{region_id}:carbon_offset"
    s_net = (
        (vo[s_net_col] - vo[s_offset_col]) / 1e9 if s_net_col in vo and s_offset_col in vo else None
    )

    if s_bau is not None:
        ax.plot(years, s_bau / 1e9, color="#E0A800", linewidth=1.0, alpha=0.6)
    if s_net is not None:
        ax.plot(years, s_net, color="#757575", linewidth=1.2)

    for top_key, bottom_key, color, alpha, label in fills:
        top = vo.get(f"{region_id}:{top_key}")
        bottom = vo.get(f"{region_id}:{bottom_key}")
        if top is not None and bottom is not None:
            ax.fill_between(
                years,
                top / 1e9,
                bottom / 1e9,
                color=color,
                alpha=alpha,
                label=label,
                edgecolor=color,
                linewidth=0.3,
            )

    ax.set_title(f"{region_id.replace('_', ' ')}", fontsize=11)
    ax.set_xlabel("Year")
    ax.set_ylabel("CO₂ Emissions (Mt)")
    ax.set_xlim(2019, 2050)
    ax.axhline(y=0, color="gray", linestyle="--", alpha=0.4, linewidth=0.8)
    ax.grid(True, alpha=0.25)

axes[0].legend(fontsize=8, loc="upper right")
plt.tight_layout()
plt.savefig(figures_dir / "eu_emissions_waterfall.pdf", format="pdf", bbox_inches="tight")
plt.show()